# 01 — Dataset exploration

Check the evaluation corpus and the calibration set before running anything expensive.

Questions this notebook answers:

1. How many tokens does the evaluation split give at the configured sequence length?
2. Do Pythia and Qwen tokenisers produce comparable sequence counts from the same text?
   (They will not — different vocabularies. That is why absolute perplexity is never
   compared across the two families.)
3. Is the calibration set disjoint from the evaluation set?
4. What is the token-stream fingerprint? Record it: two runs with different fingerprints were
   not evaluated on the same data and must not be compared.

Outputs are stripped on commit (`nbstripout` pre-commit hook), so run the cells locally.

In [ ]:
from scale_aware_compression.config import load_config
from scale_aware_compression.logging_utils import configure_logging

configure_logging("INFO")
config = load_config("../configs/experiments/pilot.yaml")
print(config.describe())
config.data

In [ ]:
from scale_aware_compression.data.preprocessing import chunk_sequence, fingerprint_token_ids

# Both functions are implemented and torch-free, so this cell runs without a model.
token_ids = list(range(10_000))
blocks = chunk_sequence(token_ids, config.data.sequence_length)
print(f"{len(token_ids)} tokens -> {len(blocks)} blocks of {config.data.sequence_length}")
print(f"fingerprint: {fingerprint_token_ids(token_ids)}")

In [ ]:
from scale_aware_compression.data.calibration import select_calibration_indices

# Derived from the fixed calibration seed, not the run seed: varying the run seed for error
# bars must not change which sequences the quantiser calibrates on.
indices = select_calibration_indices(
    population_size=len(blocks),
    num_samples=config.data.calibration_samples,
    seed=config.data.calibration_seed,
)
print(f"{len(indices)} calibration sequences, first ten: {indices[:10]}")

again = select_calibration_indices(
    population_size=len(blocks),
    num_samples=config.data.calibration_samples,
    seed=config.data.calibration_seed,
)
assert indices == again, "calibration selection must be deterministic"

## Still to do

Once `data/loaders.py` is implemented:

- load the real evaluation split and print its `DatasetSummary`
- tokenise the same text with both the Pythia and Qwen tokenisers and compare token counts
- plot the document-length distribution, to see how much text the final-block drop discards
- assert the calibration and evaluation index sets are disjoint